In [31]:
from transformers import AutoImageProcessor, AutoModelForImageClassification,AutoConfig
import pytorch_lightning as pl
from torchmetrics import Accuracy
import torch
from transformers import ViTImageProcessor, ViTForImageClassification
from torch.utils.data import DataLoader
config = AutoConfig.from_pretrained("rizvandwiki/gender-classification")

processor = AutoImageProcessor.from_pretrained("rizvandwiki/gender-classification")
model = AutoModelForImageClassification.from_pretrained("rizvandwiki/gender-classification")



Loading weights: 100%|██████████| 200/200 [00:00<00:00, 6237.72it/s]


In [32]:
class Classifier(pl.LightningModule):

    def __init__(self, model, lr: float = 2e-5, **kwargs):
        super().__init__()
        self.save_hyperparameters('lr', *list(kwargs))
        self.model = model
        self.forward = self.model.forward
        self.val_acc = Accuracy(
            task='multiclass' if model.config.num_labels > 2 else 'binary',
            num_classes=model.config.num_labels
        )

    def training_step(self, batch, batch_idx):
        outputs = self(**batch)
        self.log(f"train_loss", outputs.loss)
        return outputs.loss

    def validation_step(self, batch, batch_idx):
        outputs = self(**batch)
        self.log(f"val_loss", outputs.loss)
        acc = self.val_acc(outputs.logits.argmax(1), batch['labels'])
        self.log(f"val_acc", acc, prog_bar=True)
        return outputs.loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


In [33]:
import os
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as transforms

ages={range(0, 3):0,range(4, 7):1,range(8, 14):2,range(15, 21):3,range(25, 33):4,
              range(38, 44):5,range(48, 54):6,range(60, 101):7}
genders={'f':0,'m':1}
class CustomImageDataset(Dataset):
    def __init__(self, img_dir, txt_file, transform=None):
        """
        img_dir: 图片存放的文件夹路径
        txt_file: txt 文件的完整路径
        transform: 数据增强（可选）
        """
        self.img_dir = img_dir
        self.transform = transform
        
        
        self.samples = []
        with open(txt_file, 'r') as f:
            next(f)
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()          # 用空格分割
                subdir_name=parts[0]
                img_name = "landmark_aligned_face."+parts[2]+"."+parts[1]
                if(parts[3]=="None"):
                    continue
                if(parts[4][-1]==')'):
                    age=int(parts[4][:-1])
                    gender=parts[5]
                else:
                    age=int(parts[3])
                    gender=parts[4]
                for k in list(ages.keys()):
                    if(age in k):
                        age=ages[k]
                        break
                if(gender not in ['f','m']):
                    continue
                img_path = os.path.join(img_dir,subdir_name, img_name)
                if os.path.exists(img_path):  # 防止文件不存在
                    self.samples.append((img_path, age,genders[gender]))
        
        # 自动统计类别信息（方便后面使用）
        self.classes = [0,1]
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        
        print(f"Dataset loaded! A total of {len(self.samples)} images, {len(self.classes)} classes")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, age,gender = self.samples[idx]
        
        # 打开图片
        image = Image.open(img_path).convert('RGB')
        
        # 应用 transform（Resize、ToTensor、Normalize 等）
        if self.transform:
            image = self.transform(image)
        
        return image, gender
class EmptyDataset(Dataset):
    def __init__(self):
        pass   # 什么都不用做
    
    def __len__(self):
        return 0
    
    def __getitem__(self, index):
        # 永远不会被调用，因为 len=0
        raise IndexError("Empty dataset cannot be indexed")

In [34]:
class ImageClassificationCollator:
    def __init__(self, feature_extractor):
        self.feature_extractor = feature_extractor

    def __call__(self, batch):
        encodings = self.feature_extractor([x[0] for x in batch], return_tensors='pt')
        encodings['labels'] = torch.tensor([x[1] for x in batch], dtype=torch.long)
        return encodings
feature_extractor = ViTImageProcessor.from_pretrained('rizvandwiki/gender-classification')


In [36]:
from pathlib import  Path
import math
from torch.utils.data import ConcatDataset
txt_dir="/Users/gongjianxiang/programming/python/deep learning/Project/label txt"
folder = Path(txt_dir)
paths = [str(p) for p in folder.iterdir()]
K = len(paths)
avg_acc=0
for i in range(K):
    train_ds=EmptyDataset()
    val_ds=CustomImageDataset(img_dir="/Users/gongjianxiang/programming/python/deep learning/Project/aligned",
                           txt_file=paths[i])
    for p in paths[:i]+paths[i+1:]:
        train_ds=ConcatDataset([train_ds,CustomImageDataset(img_dir="/Users/gongjianxiang/programming/python/deep learning/Project/aligned",
                           txt_file=p)])
        
    collator = ImageClassificationCollator(feature_extractor)
    train_loader = DataLoader(train_ds, batch_size=128, collate_fn=collator, num_workers=0, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=128, collate_fn=collator, num_workers=0)

    """pl.seed_everything(42)
    classifier = Classifier(model, lr=2e-5)
    trainer = pl.Trainer(accelerator='gpu', devices=1, precision=16, max_epochs=4)
    trainer.fit(classifier, train_loader, val_loader)"""
    val_batch = next(iter(val_loader))
    outputs = model(**val_batch).logits.argmax(dim=1)
    print('Preds: ', outputs)
    print('Labels:', val_batch['labels'])
    print("Accuracy: ",(outputs == val_batch['labels']).float().mean())
    avg_acc+=(outputs == val_batch['labels']).float().mean()
avg_acc/=K
print("Average accuracy is",avg_acc) 


Dataset loaded! A total of 3597 images, 2 classes
Dataset loaded! A total of 3995 images, 2 classes
Dataset loaded! A total of 3445 images, 2 classes
Dataset loaded! A total of 3124 images, 2 classes
Dataset loaded! A total of 3291 images, 2 classes
Preds:  tensor([0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1,
        0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1,
        1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0,
        0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1,
        0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
        1, 0, 1, 0, 0, 0, 1, 1])
Labels: tensor([0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1,
        0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1,
        1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0,
        0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0,